# 03b · Gerçek NAS: evrimsel arama (`nas_search1`)

**Girdi:** 03'te eğitilmiş süper ağın son doğrulanmış kaydı (`nas_super1`, 60 epoch; uzatıldıysa 90) + Jetson Orin Nano CPU'da ölçülmüş gecikme tablosu (LUT; RPi 5 gelince RPi LUT'u ile tekrar). **Çıktı:** NSGA-II ile bulunan mAP–gecikme Pareto cephesi ve 03c'nin sıfırdan 60 epoch eğiteceği **en çok 11 aday** (≤ 3 Pareto + 7 gecikmeye göre katmanlı + Nano@416).

**Arama (DetNAS ölçeği):** NSGA-II, nüfus 50 × 20 nesil ≈ 1000 alt ağ; amaçlar: uygunluk ↑ = val2017 **A_fit1k** mAP@[.5:.95] (0–100), süper ağ EMA ağırlıkları, BN istatistikleri her alt ağ için train2017 bn2k (2000) ile yeniden hesaplanır; gecikme ↓ = LUT tahmini (doğrulamadan geçen doğrusal düzeltmeyle). Başlangıç nüfusuna Nano (7 çözünürlükte), en büyük ve en küçük alt ağ konur. **val2017-B mühürlü**: bu notebook B'yi hiçbir yükleyiciye vermez.

**Önceden kayıtlı kapılar ve kurallar (manifestte):**
- LUT kapısı: uçtan uca doğrulamada Spearman ρ ≥ 0,9 değilse arama başlamaz (konuşulur).
- Aday seçimi (`scripts/rank_select.py`; Furkan 2026-10-17): Pareto — her hedef r ∈ {1,00; 0,80; 0,65} için LUT ≤ r · T_N olanlar arasında en yüksek fit1k; aynı mimari birden çok hedefi karşılarsa tekilleştirilir (yerine aday konmaz), boş hedef gevşetilmez, 1,00 hedefi yalnız aday toplar; katmanlı — kalanlar LUT'a göre 7 eşit dilim, dilim başına tohumlu (2026) bir aday; + Nano@416 → en çok 11 aday.
- Seçilen 11 adayın süper ağ skoru ayrıca **tam val2017-A**'da (2500) ölçülür → 03c'deki Kendall τ'nun süper ağ tarafı.

**Kesintiye dayanıklılık:** NSGA-II durumu her değerlendirmeden sonra Drive'a atomik yazılır (`search_state.json`); yeniden çalıştırınca değerlendirilmiş alt ağlar tekrar ölçülmez, kaldığı nesilden devam eder. Seçim ve A skorları da durum dosyasından tekrar üretilir.

**Süre/maliyet (tahmin):** veri ≈ 5 dk (tam train2017 indirilmez: yalnız bn2k'nın 2000 görseli), arama ≈ 1000 × 25 sn ≈ 7 sa ≈ 11 CU, A skorları ≈ 11 × 60 sn.

**Kayıtlar** (`Drive edge_ai/results/nas_search1/`): `manifest.json` · `search_state.json` · `evaluated.csv` · `pareto.csv` · `candidates.json` · `nas_search1_pareto.png` · `nas_search1_summary.json`.

Kod: `scripts/supernet_search.py` (değerlendirme, BN kalibrasyonu, NSGA-II, LatencyModel), `scripts/rank_select.py`, `exps/nas/supernet.py`.

In [ ]:
# 1 · Ortam ve sürüm kaydı
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

In [ ]:
# 2 · Kod: depo + YOLOX fork sabit commit (exp ve betik depoda: exps/nas, scripts)
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
else:  # runtime aynı kaldıysa depodaki son exp/betik sürümünü al
    subprocess.run(["git", "-C", WS, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)
env["repo_dirty"] = bool(sh(f"git -C {WS} status --porcelain"))
assert not env["repo_dirty"], "depo kirli: kayıtlı commit koşulan kodu temsil etmez"

In [ ]:
# 3 · YOLOX bağımlılıkları
# Not: `pip install -e .` (setup.py develop) güncel setuptools + py3.13'te kırılıyor; YOLOX saf Python
# olduğu için kurulum yerine PYTHONPATH kullanılır. onnx-simplifier eğitimde gerekmez → atlandı.
# onnx + onnxruntime: check_supernet.py'nin ONNX opset 13 eşdeğerlik denetimi için (hücre 7).
!pip install -q loguru tabulate thop ninja pycocotools psutil tensorboard opencv-python-headless onnx onnxruntime

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "pycocotools", "loguru", "thop", "onnx", "onnxruntime"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
from yolox.exp import get_exp
exp = get_exp(None, "yolox-nano")
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})
SCRIPTS = f"{WS}/scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
NAS_EXPS = f"{WS}/exps/nas"
if NAS_EXPS not in sys.path:
    sys.path.insert(0, NAS_EXPS)
import importlib, nas_search as ns, nas_data as nd, supernet as sn, supernet_search as ss
for _m in (ns, nd, sn, ss):
    importlib.reload(_m)
EXP_FILE = f"{WS}/exps/nas/supernet.py"

In [ ]:
# 4 · Drive + ağırlık (SHA256)
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
env.pop("pkg_yolox", None)
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/weights"
ROUND = "nas_search1"   # gerçek NAS, evrimsel arama (Karar 6)
SUPER = f"{DRIVE_ROOT}/results/nas_super1"   # 03 çıktıları (salt okunur)
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/{ROUND}"
for d in (DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

import hashlib, urllib.request, shutil
WEIGHT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
w_drive = f"{DRIVE_WEIGHTS}/yolox_nano.pth"
if not os.path.exists(w_drive):
    urllib.request.urlretrieve(WEIGHT_URL, w_drive)
WEIGHTS = "/content/yolox_nano.pth"
shutil.copy(w_drive, WEIGHTS)

h = hashlib.sha256()
with open(WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
env["weights_url"] = WEIGHT_URL
env["weights_sha256"] = h.hexdigest()
env["weights_bytes"] = os.path.getsize(WEIGHTS)
print(env["weights_sha256"], env["weights_bytes"])

In [ ]:
# 5 · Veri (hafif): anotasyonlar + val2017 + train2017'den yalnız bn2k'nın 2000 görseli
# Arama tam train2017 gerektirmez: uygunluk val2017-A_fit1k'da, BN kalibrasyonu bn2k'da. bn2k görselleri COCO'nun
# resmi görsel adreslerinden (anotasyondaki coco_url) tek tek indirilir; eksik/bozuk olan yeniden denenir.
import zipfile, shutil, time, urllib.request, concurrent.futures as cf
COCO_LOCAL = "/content/datasets/COCO"
ANN_ZIP, VAL_ZIP = f"{DRIVE_DATA}/annotations_trainval2017.zip", f"{DRIVE_DATA}/val2017.zip"
t0 = time.time()
os.makedirs(f"{COCO_LOCAL}/annotations", exist_ok=True)
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in ("annotations/instances_train2017.json", "annotations/instances_val2017.json"):
        if not os.path.exists(f"{COCO_LOCAL}/{name}"):
            z.extract(name, COCO_LOCAL)
if not os.path.isdir(f"{COCO_LOCAL}/val2017"):
    shutil.copy(VAL_ZIP, "/content/val2017.zip")
    with zipfile.ZipFile("/content/val2017.zip") as z:
        z.extractall(COCO_LOCAL)
env["train_ann_sha256"] = ns.sha256(f"{COCO_LOCAL}/annotations/instances_train2017.json")
env["val_images"] = len(os.listdir(f"{COCO_LOCAL}/val2017"))
assert env["val_images"] == 5000
print({k: env[k] for k in ("train_ann_sha256", "val_images")}, f"{time.time() - t0:.0f} sn")

In [ ]:
# 6 · Veri ayrımları (03 ile birebir aynı olmalı) + bn2k görselleri
SPLIT_SEED = 2026
sm_local = f"{COCO_LOCAL}/annotations/nas_splits_manifest.json"
if not os.path.exists(sm_local):
    r = subprocess.run(["python", f"{WS}/scripts/make_splits.py", "--coco", COCO_LOCAL, "--seed", str(SPLIT_SEED)],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:]); assert r.returncode == 0
sm = json.load(open(sm_local))
sm_super = ns.read_json(f"{SUPER}/splits_manifest.json")
assert sm["ids_sha256"] == sm_super["ids_sha256"] and \
    {k: v["sha256"] for k, v in sm["files"].items()} == {k: v["sha256"] for k, v in sm_super["files"].items()}, \
    "ayrım 03'tekinden farklı"
bn = json.load(open(f"{COCO_LOCAL}/annotations/instances_train2017_bn2k.json"))
os.makedirs(f"{COCO_LOCAL}/train2017", exist_ok=True)
def fetch(im):
    dst = f"{COCO_LOCAL}/train2017/{im['file_name']}"
    for attempt in range(5):
        try:
            if os.path.exists(dst) and os.path.getsize(dst) > 0:
                import cv2
                if cv2.imread(dst) is not None:
                    return True
            urllib.request.urlretrieve(im["coco_url"], dst + ".part"); os.replace(dst + ".part", dst)
        except Exception as e:
            time.sleep(2 * (attempt + 1))
    return False
t0 = time.time()
with cf.ThreadPoolExecutor(32) as ex:
    ok = list(ex.map(fetch, bn["images"]))
assert all(ok), f"{ok.count(False)} bn2k görseli indirilemedi (hücreyi yeniden çalıştır)"
env["bn2k_images"] = len(bn["images"])
print(f"bn2k: {len(bn['images'])} görsel hazır ({time.time() - t0:.0f} sn) · ayrım 03 ile aynı")
# MÜHÜR: instances_val2017_B.json hiçbir yükleyiciye verilmez.

In [ ]:
# 7 · Girdiler: süper ağ (son doğrulanmış kayıt) + LUT (kapı: Spearman ρ ≥ 0,9)
import torch
from ckpt_store import CheckpointStore
import rank_select as rs
importlib.reload(rs)
sup_manifest = ns.read_json(f"{SUPER}/manifest.json")
sup_status = ns.read_json(f"{SUPER}/runs/supernet/status.json")
print("süper ağ:", sup_status["status"], "| uzatma kararı:", sup_status.get("extension_verdict"))
assert sup_status["status"] in ("trained", "evaluated"), "süper ağ eğitimi bitmemiş"
SUPER_FINAL_EPOCH = 60   # ← uzatma yapıldıysa 90 (Furkan onayı + 03 hücre 10 kararı)
IPE = 118_287 // 64
store = CheckpointStore(f"{SUPER}/runs/supernet/ckpt", "/content/ck_local_search", log=print)
state, rec = store.find(SUPER_FINAL_EPOCH * IPE)
assert state is not None, f"epoch {SUPER_FINAL_EPOCH} kaydı yok"
env["supernet_record"] = {"file": rec["file"], "sha256": rec["sha256"], "progress": rec["progress"]}
exp0 = sn.Exp(); SUP = exp0.get_model(); SUP.load_state_dict(state["model"], strict=True); SUP = SUP.cuda().eval()
NAMES = exp0.dw_names; N = len(NAMES)
del state
LUT_DEVICE = "jetson_orin_nano_cpu"   # RPi 5 gelince "rpi5_cpu" ile ayrı tur (nas_search1_rpi)
LUT_DIR = f"{SUPER}/lut/{LUT_DEVICE}"
lut_path, val_path = f"{LUT_DIR}/lut.json", f"{LUT_DIR}/lut_validation.json"
assert os.path.exists(lut_path), f"{lut_path} yok: Jetson ölçümü + fit (scripts/latency_lut.py) gerekli"
lv = ns.read_json(val_path)
print("LUT doğrulama:", {k: lv[k] for k in ("n_val", "spearman_raw", "mape_raw_pct", "mape_calibrated_pct", "calib")})
LUT_GATE = 0.9
assert lv["spearman_raw"] >= LUT_GATE, "LUT kapısı geçmedi (ρ < 0,9): arama başlamaz, konuşulmalı"
LAT = ss.LatencyModel.load(lut_path)
NANO_ID = sn.config_id(sn.nano_config(N, 416))
print("Nano@416 LUT gecikmesi:", round(LAT(sn.nano_config(N, 416), NAMES), 3), "ms")
EV = ss.Evaluator(COCO_LOCAL, batch_size=64, workers=os.cpu_count())

In [ ]:
# 8 · Manifest (dondurulmuş): arama tarifi, girdilerin hash'leri, aday seçim kuralı
POP, GENS, SEARCH_SEED, SELECT_SEED = 50, 20, 0, 2026
SEEDS = [sn.nano_config(N, r) for r in sn.RESOLUTIONS] + [sn.max_config(N), sn.min_config(N)]
FINGERPRINT = ("round", "supernet_record", "lut", "search", "fitness", "selection_rule", "splits", "yolox_commit")
manifest = ns.ensure_manifest(f"{DRIVE_RESULTS}/manifest.json", {
    "round": ROUND, "notebook": "notebooks/03b_gercek_nas_evrimsel_arama.ipynb",
    "supernet_record": env["supernet_record"], "supernet_manifest_created": sup_manifest["created_utc"],
    "lut": {"device": LUT_DEVICE, "sha256": ns.sha256(lut_path), "validation": {k: lv[k] for k in (
        "n_val", "spearman_raw", "mape_raw_pct", "mape_calibrated_pct", "calib")}, "gate": f"spearman_raw ≥ {LUT_GATE}"},
    "search": {"algorithm": "NSGA-II (Deb 2002)", "population": POP, "generations": GENS, "seed": SEARCH_SEED,
               "children_per_generation": POP // 2, "mutation": {"p_kernel": 0.1, "p_depth": 0.2, "p_res": 0.2},
               "crossover": "düzgün (her gen iki ebeveynden eşit olasılıkla)", "objectives": ["fit1k mAP ↑", "LUT ms ↓"],
               "seeds": [sn.config_id(c) for c in SEEDS]},
    "fitness": {"split": "val2017 A_fit1k (1000)", "bn": "train2017 bn2k (2000), kümülatif", "weights": "süper ağ EMA",
                "precision": "fp16", "conf": 0.001, "nms": 0.65, "ap_scale": "0–100"},
    "selection_rule": {"module": "scripts/rank_select.py", "pareto_targets_x_nano": list(rs.PARETO_TARGETS),
                       "pareto": "her hedef r için LUT ≤ r · T_N olanlarda en yüksek fit1k; aynı mimari birden çok hedefi karşılarsa "
                                 "tekilleştirilir (yerine aday konmaz); boş hedef gevşetilmez; 1,00 hedefi yalnız aday toplar",
                       "stratified": f"kalanlar LUT'a göre {rs.N_STRATIFIED} eşit dilim, dilim başına tohumlu (seed "
                                     f"{SELECT_SEED}) bir aday", "always": "Nano@416", "total": "en çok 3 + 7 + 1 = 11 (tekilleştirmeyle daha az)",
                       "decided_by": "Furkan, 2026-10-17 (deney tasarımı tercihi; literatür eşiği değil)",
                       "supernet_score_for_tau": "seçilenlerin tam val2017-A (2500) mAP'i, BN kalibreli"},
    "splits": {"ids_sha256": sm["ids_sha256"], "sealed": "val2017 B açılmaz"},
    "yolox_commit": env["yolox_commit"], "repo_commit_at_creation": env["repo_commit"], "env_at_creation": env,
}, keys=FINGERPRINT)
print(json.dumps({k: manifest[k] for k in ("round", "created_utc", "search", "selection_rule")}, indent=1, default=str))

In [ ]:
# 9 · NSGA-II (UZUN KOŞU ≈ 7 sa — Furkan'ın başlatma onayıyla). Kesintiden sonra yeniden çalıştır: kaldığı yerden.
STATE = f"{DRIVE_RESULTS}/search_state.json"
t0 = time.time()
def fitness(cfg):
    return ss.evaluate_config(SUP, NAMES, cfg, EV, log=lambda s: None)
def log(s):
    print(f"[{time.time() - t0:6.0f}s] {s}", flush=True)
st = ss.evolve(STATE, NAMES, fitness, LAT, pop_size=POP, generations=GENS, seed=SEARCH_SEED, seeds=SEEDS, log=log)
print("değerlendirilen:", len(st["evaluated"]), "| nesil:", st["generation"], "| son:", st["history"][-1] if st["history"] else None)

In [ ]:
# 10 · Pareto + aday seçimi (önceden kayıtlı kural) + seçilenlerin tam val2017-A süper ağ skoru
import csv
st = ns.read_json(STATE)
ev = st["evaluated"]
assert st["generation"] == GENS, "arama bitmemiş"
assert NANO_ID in ev, "Nano@416 değerlendirilmemiş (başlangıç nüfusunda olmalıydı)"
front = ss.pareto(st)
def dump(path, ids, extra=None):
    with open(path + ".tmp", "w", newline="") as f:
        w = csv.writer(f); w.writerow(["config_id", "map5095_fit1k", "lut_ms", "on_front"])
        for k in ids:
            w.writerow([k, ev[k]["map5095"], ev[k]["latency_ms"], k in front])
    os.replace(path + ".tmp", path)
dump(f"{DRIVE_RESULTS}/evaluated.csv", sorted(ev, key=lambda k: ev[k]["latency_ms"]))
dump(f"{DRIVE_RESULTS}/pareto.csv", front)
cpath = f"{DRIVE_RESULTS}/candidates.json"
cands = ns.read_json(cpath)
if cands is None:
    sel, sel_notes = rs.select_candidates(ev, NANO_ID, seed=SELECT_SEED)
    cands = {"selected": sel, "notes": sel_notes, "rule": manifest["selection_rule"], "selected_utc": ns.now_utc(),
             "a_scores": {}}
    ns.write_json(cpath, cands)
DEV = ss.Evaluator(COCO_LOCAL, fit_ann=ss.DEV_ANN, batch_size=64, workers=os.cpu_count())
for c in cands["selected"]:
    k = c["config_id"]
    if k and k not in cands["a_scores"]:
        cands["a_scores"][k] = ss.evaluate_config(SUP, NAMES, sn.parse_config_id(k), DEV, log=lambda s: None)
        ns.write_json(cpath, cands)
for c in cands["selected"]:
    k = c["config_id"]
    print(f"{c['group']:10s} {str(c.get('lut_ratio')):>7s} × Nano · fit1k {c.get('fit1k')} · A {cands['a_scores'].get(k, {}).get('map5095')} · {k}")
print("cephe büyüklüğü:", len(front), "| değerlendirilen:", len(ev), "| notlar:", cands["notes"])

In [ ]:
# 11 · Özet + grafik → nas_search1_summary.json, nas_search1_pareto.png
import matplotlib.pyplot as plt
xs = [ev[k]["latency_ms"] for k in ev]; ys = [ev[k]["map5095"] for k in ev]
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(xs, ys, s=6, alpha=0.35, label=f"değerlendirilen ({len(ev)})")
fx = sorted(front, key=lambda k: ev[k]["latency_ms"])
ax.plot([ev[k]["latency_ms"] for k in fx], [ev[k]["map5095"] for k in fx], "-o", ms=3, label="Pareto cephesi")
for c in cands["selected"]:
    if c["config_id"]:
        k = c["config_id"]; mk = {"pareto": "*", "stratified": "s", "nano": "D"}[c["group"]]
        ax.scatter(ev[k]["latency_ms"], ev[k]["map5095"], marker=mk, s=90, edgecolor="k", zorder=5)
ax.set_xlabel(f"LUT gecikmesi (ms, {LUT_DEVICE}, ORT CPU 4 thread)"); ax.set_ylabel("mAP@[.5:.95] val2017 A_fit1k (süper ağ)")
ax.set_title("nas_search1: NSGA-II (★ Pareto seçimi, ■ katmanlı, ◆ Nano@416)"); ax.legend(); fig.tight_layout()
fig.savefig(f"{DRIVE_RESULTS}/{ROUND}_pareto.png", dpi=150); plt.show()
ns.write_json(f"{DRIVE_RESULTS}/{ROUND}_summary.json", {
    "round": ROUND, "written_utc": ns.now_utc(), "evaluated": len(ev), "front": len(front),
    "history": st["history"], "selected": cands["selected"], "a_scores": {k: v["map5095"] for k, v in cands["a_scores"].items()},
    "wall_s_this_session": round(time.time() - t0)})